In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
pd.options.display.float_format = '{:.2f}'.format
warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [2]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по овцы и козы v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Овцы и козы
684,ГАЛМАТЫ,2019-02-01,0.40
1776,ОБЛАСТЬ ЖЕТІСУ,2022-10-01,3002.13
463,АТЫРАУСКАЯ ОБЛАСТЬ,2021-11-01,1051.37
820,ГАСТАНА,2020-02-01,1.50
565,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2019-10-01,2288.86
24,АКМОЛИНСКАЯ ОБЛАСТЬ,2017-01-01,625.40
1516,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2018-01-01,547.30
495,АТЫРАУСКАЯ ОБЛАСТЬ,2024-07-01,734.44
2099,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2025-05-01,431.65
960,ГШЫМКЕНТ,2024-08-01,43.30


In [3]:
regions = df['Регион'].unique()
target   = "Овцы и козы"
horizon  = 3
epsilon = 1e-6

In [4]:
first_test = pd.to_datetime("2024-08-01")
last_possible = df["Период"].max() - pd.DateOffset(months=horizon-1)
test_starts = pd.date_range(first_test, last_possible, freq="MS")

## Holt-Winter's (log)

In [5]:
results_hw = []
for region in regions:
    ts = (df[df["Регион"] == region]
          .set_index("Период")[target]
          .dropna()
          .sort_index())
    if len(ts) < 24:
        print(f"{region}: всего {len(ts)} мес. — сезонный Holt-Winter's невозможен.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        # формируем train / test
        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]

        # пропускаем, если недостаточно данных или неполный test
        if len(train) < 24 or len(test) < horizon:
            continue

        # обучаем модель
        train_log = np.log1p(train)

        hw_log = ExponentialSmoothing(
            train_log,
            seasonal="add",
            seasonal_periods=12
        ).fit(optimized=True)

        # прогноз и метрики
        fc_log = hw_log.forecast(horizon)
        fc = np.expm1(fc_log) 
        # fc   = model.forecast(horizon)
        rmse = np.sqrt(mean_squared_error(test, fc))
        mae  = mean_absolute_error(test, fc)
        mape = (np.abs((test - fc) / test).mean()) * 100

        results_hw.append({
            "Регион":      region,
            "Test start":  test_start.strftime("%Y-%m"),
            "Test end":    test_end.strftime("%Y-%m"),
            "Forecast":    [x.round(2) for x in list(fc.values)],
            "Actual":      [y.round(2) for y in list(test.values)],
            "RMSE":        rmse,
            "MAE":         mae,
            "MAPE_%":      mape
        })

# 4) Усреднение по всем скользящим окнам для каждого региона
res_hw = pd.DataFrame(results_hw)
res_hw.to_excel("results/Овцы и козы - Результаты прогнозов ХВ v2.xlsx", index=False)
print("Результаты прогнозов HW на 3 месяца:")

display(res_hw)

final_hw = (
    res_hw
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_hw.to_excel("results/Овцы и козы - Результаты прогнозов ХВ средние v2.xlsx", index=False)
print("Средние метрики Holt–Winter's по регионам (rolling-3):")
display(final_hw)

Результаты прогнозов HW на 3 месяца:


,Регион,Test start,Test end,Forecast,Actual,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"[475.79, 739.34, 773.79]","[495.67, 679.44, 710.13]",51.76,47.81,7.26
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"[742.36, 777.1, 809.27]","[679.44, 710.13, 843.07]",56.53,54.56,7.57
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"[770.29, 803.99, 648.14]","[710.13, 843.07, 717.2]",57.49,56.10,7.58
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"[800.21, 647.9, 556.45]","[843.07, 717.2, 606.56]",55.23,54.09,7.67
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"[649.1, 557.46, 618.78]","[717.2, 606.56, 807.76]",119.39,102.06,13.66
...,...,...,...,...,...,...,...,...
189,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"[4502.39, 4573.18, 4687.51]","[4447.97, 4266.88, 4540.22]",198.73,169.34,3.88
190,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"[4556.5, 4671.23, 3923.52]","[4266.88, 4540.22, 3939.72]",183.76,145.61,3.36
191,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"[4590.2, 3854.92, 3758.32]","[4540.22, 3939.72, 3631.9]",92.50,87.07,2.24
192,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"[3843.92, 3747.57, 4663.25]","[3939.72, 3631.9, 3915.0]",440.62,319.91,8.24


Средние метрики Holt–Winter's по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,82.03,76.07,10.06
1,АКТЮБИНСКАЯ ОБЛАСТЬ,139.36,97.03,4.04
2,АЛМАТИНСКАЯ ОБЛАСТЬ,350.71,288.23,11.60
3,АТЫРАУСКАЯ ОБЛАСТЬ,103.65,76.98,7.38
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,148.50,110.22,6.68
5,ГАЛМАТЫ,0.60,0.53,NaN
6,ГАСТАНА,0.14,0.13,14.25
7,ГШЫМКЕНТ,12.76,10.31,12.97
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,119.88,104.40,3.40
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,22.72,18.57,1.59


## SARIMA

In [6]:
results_sarima = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    ts = ts + epsilon
    ts_log = np.log(ts)

    if len(ts_log) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. для авто-ARIMA, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train_log = ts_log[ts_log.index < test_start]
        test_log  = ts_log[(ts_log.index >= test_start) & (ts_log.index <= test_end)]
        if len(train_log) < 12 + horizon or len(test_log) < horizon:
            continue

        # автоподбор на лог-данных
        use_seasonal = len(train_log) >= 2 * 12

        sarima_log = auto_arima(
            train_log,
            seasonal=use_seasonal,
            m=12 if use_seasonal else 1,
            D=1 if use_seasonal else 0,      # фиксируем порядок сезонной разности
            seasonal_test=None,               # пропустить nsdiffs
            boxcox=True,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore"
        )
      
        # прогноз в лог-шкале
        fc_log = sarima_log.predict(n_periods=horizon, return_conf_int=False)

        # возвращаем прогноз в исходные единицы
        fc = np.exp(fc_log) - epsilon
        actual = np.exp(test_log.values) - epsilon  # но exp(log(x)) == x

        # метрики на исходном уровне
        rmse = np.sqrt(mean_squared_error(actual, fc))
        mae  = mean_absolute_error(actual, fc)
        mape = (np.abs((actual - fc) / actual).mean()) * 100

        results_sarima.append({
            "Регион":         region,
            "Test start":     test_start.strftime("%Y-%m"),
            "Test end":       test_end.strftime("%Y-%m"),
            "order":          sarima_log.order,
            "seasonal_order": sarima_log.seasonal_order,
            "RMSE":           round(rmse,2),
            "MAE":            round(mae,2),
            "MAPE_%":         round(mape,2),
            "Forecast":       [round(x,2) for x in fc],
            "Actual":         [round(y,2) for y in actual]
        })
# формируем DataFrame с результатами
res_sarima = pd.DataFrame(results_sarima)
res_sarima.to_excel("results/Овцы и козы - Результаты прогнозов SARIMA v2.xlsx", index=False)
print("Результаты прогнозов SARIMA на 3 месяца:")

display(res_sarima)

final_sarima = (
    res_sarima
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_sarima.to_excel("results/Овцы и козы - Результаты прогнозов SARIMA средние v2.xlsx", index=False)
print("Средние метрики SARIMA по регионам (rolling-3):")
display(final_sarima)

Результаты прогнозов SARIMA на 3 месяца:


,Регион,Test start,Test end,order,seasonal_order,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"(1, 0, 0)","(1, 1, 0, 12)",40.98,40.10,6.32,"[467.26, 723.09, 758.36]","[495.67, 679.44, 710.13]"
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"(1, 0, 0)","(1, 1, 0, 12)",53.75,53.56,7.31,"[738.59, 763.61, 795.02]","[679.44, 710.13, 843.07]"
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"(1, 0, 0)","(1, 1, 0, 12)",59.92,56.38,7.48,"[741.55, 786.54, 636.03]","[710.13, 843.07, 717.2]"
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"(1, 0, 0)","(1, 1, 0, 12)",75.13,74.83,10.57,"[774.91, 633.13, 534.3]","[843.07, 717.2, 606.56]"
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"(1, 0, 0)","(1, 1, 0, 12)",136.40,116.34,15.66,"[652.0, 539.76, 590.74]","[717.2, 606.56, 807.76]"
...,...,...,...,...,...,...,...,...,...,...
189,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"(2, 1, 2)","(0, 1, 0, 12)",421.07,320.62,7.16,"[4379.68, 4460.7, 5239.99]","[4447.97, 4266.88, 4540.22]"
190,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"(2, 1, 2)","(0, 1, 0, 12)",467.04,382.36,8.70,"[4507.25, 5298.16, 3790.95]","[4266.88, 4540.22, 3939.72]"
191,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"(2, 1, 2)","(0, 1, 0, 12)",402.25,385.34,9.38,"[5088.77, 3635.38, 3328.77]","[4540.22, 3939.72, 3631.9]"
192,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"(3, 1, 3)","(0, 1, 0, 12)",599.69,596.94,15.58,"[3375.81, 3082.65, 4592.66]","[3939.72, 3631.9, 3915.0]"


Средние метрики SARIMA по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,83.69,77.20,10.12
1,АКТЮБИНСКАЯ ОБЛАСТЬ,128.68,97.54,4.41
2,АЛМАТИНСКАЯ ОБЛАСТЬ,352.25,266.76,10.21
3,АТЫРАУСКАЯ ОБЛАСТЬ,122.68,95.38,9.42
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,114.58,93.14,6.98
5,ГАЛМАТЫ,0.81,0.58,7135684506750822645760.00
6,ГАСТАНА,0.09,0.08,9.41
7,ГШЫМКЕНТ,30.49,24.08,26.15
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,125.31,107.30,3.76
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,60.56,52.58,3.39


## Facebook Prophet

In [7]:
results_prophet = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    if len(ts) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. данных, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]
        if len(train) < 12 + horizon or len(test) < horizon:
            continue

        # Подготовка данных для Prophet
#         df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
# # подготовка для одного региона
        df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
        df_prophet["y"] = np.log(df_prophet["y"] + epsilon)

        m = Prophet()
        m.fit(df_prophet)

        future = m.make_future_dataframe(periods=horizon, freq="MS")
        forecast = m.predict(future)

        # берем только прогнозные точки
        yhat_log = forecast["yhat"].values[-horizon:]
        fc = np.exp(yhat_log) - epsilon

        # m = Prophet()
        # m.fit(df_prophet)

        # # Создаем DataFrame будущих дат и делаем прогноз
        # # future = m.make_future_dataframe(periods=horizon, freq="MS")
        # # forecast = m.predict(future)

        # # Отбираем только наши горизонты
        # fc = forecast.set_index("ds")["yhat"].loc[test.index].values
        actual = test.values

        # Расчет метрик
        rmse  = np.sqrt(mean_squared_error(actual, fc))
        mae   = mean_absolute_error(actual, fc)
        mape  = (np.abs((actual - fc) / actual).mean()) * 100

        results_prophet.append({
            "Регион":     region,
            "Test start": test_start.strftime("%Y-%m"),
            "Test end":   test_end.strftime("%Y-%m"),
            "RMSE":       round(rmse, 2),
            "MAE":        round(mae, 2),
            "MAPE_%":     round(mape, 2),
            "Forecast":   [round(x, 2) for x in fc],
            "Actual":     [round(x, 2) for x in actual]
        })

# Собираем результаты в DataFrame
res_prophet = pd.DataFrame(results_prophet)
res_prophet.to_excel("results/Овцы и козы - Результаты прогнозов Prophet v2.xlsx", index=False)
print("Результаты прогнозов Prophet на 3 месяца:")
display(res_prophet)

final_prophet = (
    res_prophet
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_prophet.to_excel("results/Овцы и козы - Результаты прогнозов Prophet средние v2.xlsx", index=False)
print("Средние метрики Prophet по регионам (rolling-3):")
display(final_prophet)


17:48:32 - cmdstanpy - INFO - Chain [1] start processing
17:48:33 - cmdstanpy - INFO - Chain [1] done processing
17:48:33 - cmdstanpy - INFO - Chain [1] start processing
17:48:33 - cmdstanpy - INFO - Chain [1] done processing
17:48:34 - cmdstanpy - INFO - Chain [1] start processing
17:48:34 - cmdstanpy - INFO - Chain [1] done processing
17:48:34 - cmdstanpy - INFO - Chain [1] start processing
17:48:34 - cmdstanpy - INFO - Chain [1] done processing
17:48:34 - cmdstanpy - INFO - Chain [1] start processing
17:48:34 - cmdstanpy - INFO - Chain [1] done processing
17:48:34 - cmdstanpy - INFO - Chain [1] start processing
17:48:34 - cmdstanpy - INFO - Chain [1] done processing
17:48:35 - cmdstanpy - INFO - Chain [1] start processing
17:48:35 - cmdstanpy - INFO - Chain [1] done processing
17:48:35 - cmdstanpy - INFO - Chain [1] start processing
17:48:35 - cmdstanpy - INFO - Chain [1] done processing
17:48:35 - cmdstanpy - INFO - Chain [1] start processing
17:48:35 - cmdstanpy - INFO - Chain [1]

Результаты прогнозов Prophet на 3 месяца:


,Регион,Test start,Test end,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,70.79,57.62,8.69,"[479.03, 793.29, 667.76]","[495.67, 679.44, 710.13]"
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,84.21,78.27,10.70,"[796.53, 669.11, 766.39]","[679.44, 710.13, 843.07]"
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,57.51,53.16,6.86,"[664.6, 760.03, 686.29]","[710.13, 843.07, 717.2]"
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,51.93,45.90,6.09,"[762.84, 688.91, 577.37]","[843.07, 717.2, 606.56]"
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,127.52,89.07,11.49,"[693.07, 581.59, 589.64]","[717.2, 606.56, 807.76]"
...,...,...,...,...,...,...,...,...
189,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,278.21,237.07,5.32,"[4546.5, 4092.47, 4101.95]","[4447.97, 4266.88, 4540.22]"
190,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,343.43,325.28,7.64,"[4086.58, 4092.99, 3591.43]","[4266.88, 4540.22, 3939.72]"
191,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,373.72,365.85,8.97,"[4079.5, 3576.83, 3357.95]","[4540.22, 3939.72, 3631.9]"
192,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,208.47,194.48,5.08,"[3654.98, 3434.14, 3814.06]","[3939.72, 3631.9, 3915.0]"


Средние метрики Prophet по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,92.41,80.90,10.47
1,АКТЮБИНСКАЯ ОБЛАСТЬ,205.38,182.57,9.52
2,АЛМАТИНСКАЯ ОБЛАСТЬ,859.64,635.44,18.38
3,АТЫРАУСКАЯ ОБЛАСТЬ,133.22,102.92,8.01
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,359.24,328.34,27.39
5,ГАЛМАТЫ,0.84,0.58,NaN
6,ГАСТАНА,0.22,0.20,23.16
7,ГШЫМКЕНТ,22.50,16.04,14.11
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,270.56,235.79,7.66
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,114.87,96.05,7.25


In [8]:
# Переименуем колонки с MAPE, чтобы было понятно, к какому методу относятся
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW"})
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA"})
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet"})

# Мёрджим по региону
summary = (
    hw[["Регион", "MAPE_HW"]]
    .merge(sar[["Регион", "MAPE_SARIMA"]], on="Регион")
    .merge(pr[["Регион", "MAPE_Prophet"]], on="Регион")
)

# Определяем для каждой строки, какой столбец MAPE минимален
# idxmin вернёт название столбца с минимальным значением
summary["Best_method"] = summary[["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]] \
                           .idxmin(axis=1) \
                           .str.replace("MAPE_","")  # убираем префикс для красоты

# Если нужно, можно сразу отсортировать
# summary = summary.sort_values("Best_method")

# допустим, у вас уже есть summary
summary = summary.round({
    "MAPE_HW": 2,
    "MAPE_SARIMA": 2,
    "MAPE_Prophet": 2
})

# Готово!
print(summary.to_string(index=False))
summary.to_excel("results/Овцы и козы - Лучшие модели v2.xlsx", index=False)


                        Регион  MAPE_HW               MAPE_SARIMA  MAPE_Prophet Best_method
           АКМОЛИНСКАЯ ОБЛАСТЬ    10.06                     10.12         10.47          HW
           АКТЮБИНСКАЯ ОБЛАСТЬ     4.04                      4.41          9.52          HW
           АЛМАТИНСКАЯ ОБЛАСТЬ    11.60                     10.21         18.38      SARIMA
            АТЫРАУСКАЯ ОБЛАСТЬ     7.38                      9.42          8.01          HW
ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ     6.68                      6.98         27.39          HW
                       ГАЛМАТЫ      NaN 7135684506750822645760.00           NaN      SARIMA
                       ГАСТАНА    14.25                      9.41         23.16      SARIMA
                      ГШЫМКЕНТ    12.97                     26.15         14.11          HW
            ЖАМБЫЛСКАЯ ОБЛАСТЬ     3.40                      3.76          7.66          HW
 ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ     1.59                      3.39          7.25 

In [9]:
#FOLDER = Path("results")  # папка, где лежат файлы
FILE_HW      = "results/Овцы и козы - Результаты прогнозов ХВ средние v2.xlsx"
FILE_SARIMA  = "results/Овцы и козы - Результаты прогнозов SARIMA средние v2.xlsx"
FILE_PROPHET = "results/Овцы и козы - Результаты прогнозов Prophet средние v2.xlsx"


# === Загрузка исходных таблиц ===
final_hw      = pd.read_excel(FILE_HW)
final_sarima  = pd.read_excel(FILE_SARIMA)
final_prophet = pd.read_excel(FILE_PROPHET)

# Ожидаемые столбцы: 'Регион', 'MAPE_%', 'MAE' (и/или 'RMSE')
# Переименуем для прозрачности
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW", "MAE": "MAE_HW"})[["Регион","MAPE_HW","MAE_HW"]]
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA", "MAE": "MAE_SARIMA"})[["Регион","MAPE_SARIMA","MAE_SARIMA"]]
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet", "MAE": "MAE_Prophet"})[["Регион","MAPE_Prophet","MAE_Prophet"]]

# === Объединяем по региону ===
summary = (
    hw.merge(sar, on="Регион", how="inner")
      .merge(pr,  on="Регион", how="inner")
)


In [10]:
THRESHOLD_MAPE = 1000.0  # порог

mape_cols = ["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]
mae_cols  = ["MAE_HW","MAE_SARIMA","MAE_Prophet"]

# 1) Приведём метрики к числам (на всякий случай ещё раз)
for c in mape_cols + mae_cols:
    summary[c] = pd.to_numeric(summary[c], errors="coerce")

def choose_best_simple(row):
    # Берём числовые серии и подменяем NaN на +inf, чтобы .idxmin() стабильно работал
    mape_s = row[mape_cols].astype(float).fillna(np.inf)
    mae_s  = row[mae_cols].astype(float).fillna(np.inf)

    min_mape = mape_s.min()

    # Если все MAPE были NaN -> min = +inf
    if np.isinf(min_mape):
        criterion = "MAE"
        winner_col = mae_s.idxmin()
    elif min_mape <= THRESHOLD_MAPE:
        criterion = "MAPE"
        winner_col = mape_s.idxmin()
    else:
        criterion = "MAE"
        winner_col = mae_s.idxmin()

    method = winner_col.split("_")[-1]  # HW / SARIMA / Prophet

    return pd.Series({
        "Best_method": method,
        "Best_criterion": criterion,
        "Best_MAPE": float(mape_s.replace(np.inf, np.nan).min()),
        "Best_MAE": float(mae_s.replace(np.inf, np.nan).min())
    })

best = summary.apply(choose_best_simple, axis=1)

result = pd.concat([summary, best], axis=1)

# Округление и сохранение
for c in mape_cols + mae_cols + ["Best_MAPE","Best_MAE"]:
    result[c] = result[c].round(2)

# Если у вас есть переменная OUT_FILE — используйте её. Иначе:
OUT_FILE = "results/Овцы и козы - Лучшие модели (MAPE_then_MAE) v2.xlsx"
result.sort_values(["Best_method","Регион"]).to_excel(OUT_FILE, index=False)

result

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКМОЛИНСКАЯ ОБЛАСТЬ,10.06,76.07,10.12,77.20,10.47,80.90,HW,MAPE,10.06,76.07
1,АКТЮБИНСКАЯ ОБЛАСТЬ,4.04,97.03,4.41,97.54,9.52,182.57,HW,MAPE,4.04,97.03
2,АЛМАТИНСКАЯ ОБЛАСТЬ,11.60,288.23,10.21,266.76,18.38,635.44,SARIMA,MAPE,10.21,266.76
3,АТЫРАУСКАЯ ОБЛАСТЬ,7.38,76.98,9.42,95.38,8.01,102.92,HW,MAPE,7.38,76.98
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,6.68,110.22,6.98,93.14,27.39,328.34,HW,MAPE,6.68,93.14
5,ГАЛМАТЫ,NaN,0.53,7135684506750822645760.00,0.58,NaN,0.58,HW,MAE,7135684506750822645760.00,0.53
6,ГАСТАНА,14.25,0.13,9.41,0.08,23.16,0.20,SARIMA,MAPE,9.41,0.08
7,ГШЫМКЕНТ,12.97,10.31,26.15,24.08,14.11,16.04,HW,MAPE,12.97,10.31
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,3.40,104.40,3.76,107.30,7.66,235.79,HW,MAPE,3.40,104.40
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,1.59,18.57,3.39,52.58,7.25,96.05,HW,MAPE,1.59,18.57
